# The chapter's diffusion model in NumPy

Steps 4 to 6 of Build a Generative Model follow one training set of two 4-pixel
images through a diffusion model whose weights need no learning: the best
possible denoiser, a deterministic sampler, and classifier-free guidance. Here
you write that model as general functions. A noise schedule for any number of
steps, the one-jump formula of Step 4, the best possible denoiser for any number
of training images, a deterministic sampler over any list of steps, and the
guidance step of Step 6.

**How to work through it.** Each part asks for one or two functions. Replace
every `TODO` and the `raise NotImplementedError` under it with your code, then
run the check cell that follows. The check prints `Part N passed` when the part
is right, and stops with a message naming what is off when it is not.

## Setup

Run this cell first. It holds the chapter's two training images, two draws of noise, the two starts from pure noise, and the chapter's ten jumps.

In [ ]:
import numpy as np

LEFT = np.array([1.0, 0.0, 1.0, 0.0])            # the left column is bright
RIGHT = np.array([0.0, 1.0, 0.0, 1.0])           # the right column is bright
DATA = np.stack([LEFT, RIGHT])                   # the chapter's training set
THIRD = np.array([1.0, 1.0, 0.0, 0.0])           # used in Parts 2 and 3
IMAGES3 = np.stack([LEFT, RIGHT, THIRD])

EPS = np.array([0.13, -0.13, 0.64, 0.10])        # the chapter's draw for LEFT
EPS_NEW = np.array([0.03, 1.36, 1.22, -0.51])    # a second draw of your own

START_A = np.array([0.19, -0.52, -0.41, -2.44])  # ends at LEFT without a prompt
START_B = np.array([1.05, 1.78, -2.55, -0.14])   # ends at RIGHT without a prompt
STEPS_10 = list(range(1000, -1, -100))           # the chapter's ten jumps

## Part 1 (coding task): A noise schedule for any number of steps

Write two functions.

- `schedule(T)` returns `(betas, alpha_bar)`, two arrays of length `T`: DDPM's
  betas spread evenly from `1e-4` to `0.02`, and the signal kept after each
  step, `alpha_bar`.
- `add_noise(x0, t, eps)` returns the image after `t` steps, `sqrt(alpha_bar[t - 1])`
  times `x0` plus `sqrt(1 - alpha_bar[t - 1])` times `eps`, with step 0 the clean
  image: the signal kept after `t` steps is `alpha_bar[t - 1]`, so step 0 carries
  signal 1 and `alpha_bar[0]` is the signal after one step. It takes any number of
  steps and any step from 0 to `T`.

In [ ]:
def schedule(T):
    """The betas and the signal kept after each of T steps."""
    # TODO
    raise NotImplementedError


def add_noise(x0, t, eps):
    """The image after step t, for a clean image x0 and the noise eps."""
    # TODO
    raise NotImplementedError

In [ ]:
betas, alpha_bar = schedule(1000)
print(betas[:2], alpha_bar[:2])
assert betas.shape == alpha_bar.shape == (1000,), "both arrays should have length 1000"
assert abs(betas[0] - 1e-4) < 1e-12 and abs(betas[-1] - 0.02) < 1e-12, \
    f"the betas should run from 1e-4 to 0.02, got {betas[0]} and {betas[-1]}"
assert (alpha_bar[1:] < alpha_bar[:-1]).all() and alpha_bar[0] > 0.999, \
    "alpha_bar should start above 0.999 and fall at every step"
assert abs(alpha_bar[-1] - 4.04e-5) < 1e-6, \
    f"1000 steps should leave {alpha_bar[-1]}, which is 4.04e-05"
assert abs(schedule(200)[1][-1] - 0.1322) < 1e-3, \
    f"200 steps should leave {schedule(200)[1][-1]}, which is 0.132"

x500 = add_noise(LEFT, 500, EPS)
print(x500.round(3))
assert np.allclose(x500, [0.405, -0.125, 0.895, 0.096], atol=1e-3), \
    f"LEFT at step 500 is {x500.round(3)}, and Step 5 has (0.405, -0.125, 0.895, 0.096)"
assert np.allclose(add_noise(LEFT, 1000, EPS), [0.136, -0.130, 0.646, 0.100], atol=1e-3), \
    f"LEFT at step 1000 is {add_noise(LEFT, 1000, EPS).round(3)}, and Step 4 has (0.136, -0.130, 0.646, 0.100)"
assert np.allclose(add_noise(LEFT, 0, EPS), LEFT), \
    "at step 0 the image is clean, so the noise term should drop out"
assert np.allclose(add_noise(LEFT, 350, EPS_NEW), [0.559, 1.150, 1.566, -0.431], atol=1e-3), \
    f"LEFT at step 350 with the second draw is {add_noise(LEFT, 350, EPS_NEW).round(3)}"
print("Part 1 passed")

## Part 2 (coding task): The best possible denoiser

Write `best_guess(xt, t, images)`, the clean-image estimate of the best
network that could be trained on `images`: the weighted average of the training
images, each weighted by how likely it is to have produced `xt`. Step 5's
`estimate` function is the case of two images.

Return two things:

- `estimate`, the average, of the same length as `xt`
- `weights`, one weight per training image, adding up to 1

A training image at distance $d$ from `xt` gets weight
$e^{-d} / \sum e^{-d}$, with
$d = \lVert xt - \sqrt{\alpha\_bar(t - 1)} \cdot (\text{image}) \rVert^2 / (2 (1 - \alpha\_bar(t - 1)))$
at step `t`. Handle any number of training images, and any step.

In [ ]:
def best_guess(xt, t, images):
    """The best possible denoiser: (estimate, weights) for the training set images."""
    # TODO
    raise NotImplementedError

In [ ]:
guess, weights = best_guess([0.405, -0.125, 0.895, 0.096], 500, DATA)
print(guess.round(3), weights.round(3))
assert weights.shape == (2,), "there should be one weight per training image"
assert np.isclose(weights.sum(), 1), f"the weights add up to {weights.sum()}"
assert np.allclose(guess, [0.6, 0.4, 0.6, 0.4], atol=1e-3), \
    f"at step 500 the estimate is {guess.round(3)}, and Step 5 has (0.6, 0.4, 0.6, 0.4)"

x500 = add_noise(LEFT, 500, EPS)
assert np.isclose(best_guess(x500, 200, DATA)[1][0], 0.959, atol=1e-3), \
    f"read as step 200 the weight on LEFT is {best_guess(x500, 200, DATA)[1][0]:.3f}, and Step 5 has 0.959"
assert np.isclose(best_guess(x500, 350, DATA)[1][0], 0.730, atol=1e-3), \
    f"read as step 350 the weight on LEFT is {best_guess(x500, 350, DATA)[1][0]:.3f}, which should be 0.730"

three = add_noise(LEFT, 500, EPS_NEW)
guess3, weights3 = best_guess(three, 500, IMAGES3)
print(weights3.round(3), guess3.round(3))
assert weights3.shape == (3,), "a training set of three images needs three weights"
assert np.allclose(weights3, [0.369, 0.277, 0.353], atol=1e-3), \
    f"the three weights are {weights3.round(3)}"
assert np.allclose(guess3, [0.723, 0.631, 0.369, 0.277], atol=1e-3), \
    f"the estimate from three images is {guess3.round(3)}"

far = 30 * np.array([1.0, -1.0, 1.0, -1.0])
far_guess, far_weights = best_guess(far, 800, DATA)
print(far_weights.round(3))
assert not np.isnan(far_guess).any(), "a far away image should still give a clean estimate: the exponentials underflow to zero unless the smallest distance is taken out first"
assert np.allclose(far_weights, [0.991, 0.009], atol=1e-3), \
    f"a far image at step 800 is almost all LEFT, so the weights are {far_weights.round(3)}"
print("Part 2 passed")

## Part 3 (coding task): A deterministic sampler over any steps

Write `sample(start, steps, images=DATA)`, the deterministic sampler of
Step 5. Start from `start` and walk the list `steps`, which is a decreasing list
of step numbers from the noisiest to 0. At each step:

1. estimate the clean image with `best_guess`
2. read off the noise that estimate implies,
   `(x_t - sqrt(alpha_bar(t - 1)) * estimate) / sqrt(1 - alpha_bar(t - 1))`
3. jump to the next step with
   `x_next = sqrt(alpha_bar(t_next - 1)) * estimate + sqrt(1 - alpha_bar(t_next - 1)) * noise`,
   with signal 1 rather than `alpha_bar(t_next - 1)` when `t_next` is 0

Return the image at the end of the list. The list need not be evenly spaced and
need not be the chapter's ten jumps.

In [ ]:
def sample(start, steps, images=DATA):
    """The image the deterministic sampler returns after walking the steps."""
    # TODO
    raise NotImplementedError

In [ ]:
out = sample(START_A, STEPS_10)
print(out.round(3))
assert out.shape == (4,), f"the sampler should return a 4-pixel image, got {out.shape}"
assert np.allclose(out, LEFT, atol=1e-3), \
    f"start A ends at {out.round(3)}, which should be LEFT"
assert np.allclose(sample(START_B, STEPS_10), RIGHT, atol=1e-3), \
    f"start B ends at {sample(START_B, STEPS_10).round(3)}, which should be RIGHT"

one_jump = sample(START_A, [1000, 900])
print(one_jump.round(3))
assert abs(one_jump[0] - 0.195) < 1e-3, \
    f"the first pixel of the jump from 1000 to 900 is {one_jump[0]:.3f}, and Step 5 has 0.195"
assert np.allclose(sample(START_A, [1000, 0]), [0.504, 0.496, 0.504, 0.496], atol=1e-3), \
    "a single jump returns the estimate at the first step, the gray average"
assert np.allclose(sample(START_A, [1000, 750, 500, 250, 0]), [0.971, 0.029, 0.971, 0.029], atol=1e-3), \
    f"four jumps give {sample(START_A, [1000, 750, 500, 250, 0]).round(3)}"
assert np.allclose(sample(START_B, [1000, 700, 300, 0]), [0.063, 0.937, 0.063, 0.937], atol=1e-3), \
    f"uneven jumps give {sample(START_B, [1000, 700, 300, 0]).round(3)}"
assert np.allclose(sample(START_B, STEPS_10, IMAGES3), THIRD, atol=1e-3), \
    f"with three training images start B can end at the third, and it ends at {sample(START_B, STEPS_10, IMAGES3).round(3)}"
assert np.allclose(sample(np.full(4, 0.5), STEPS_10), 0.5, atol=1e-6), \
    "a start with no lean at all stays the gray average, since the estimate never favors either training image"
print("Part 3 passed")

## Part 4 (coding task): Classifier-free guidance

Write `guided_sample(start, steps, prompt_images, w, images=DATA,
negative_images=DATA)`, the same sampler with Step 6's guidance at every step.
The `images` argument is unused and is kept only to mirror Part 3's signature.

At each step the estimate becomes

```
unprompted + w * (prompted - unprompted)
```

where `unprompted` is `best_guess(x, t, negative_images)` and `prompted` is
`best_guess(x, t, prompt_images)`. `negative_images` is the training set the
unprompted network is asked about, so it defaults to the whole training set and
a negative prompt is passed by naming the images to avoid.

Return the image at the end of the list. Everything else, including the noise
the guided estimate implies, is as in Part 3.

In [ ]:
def guided_sample(start, steps, prompt_images, w, images=DATA, negative_images=DATA):
    """The image the guided sampler returns after walking the steps."""
    # TODO
    raise NotImplementedError

In [ ]:
out = guided_sample(START_B, STEPS_10, LEFT[None], 1)
print(out.round(3))
assert np.allclose(out, LEFT, atol=1e-3), \
    f"start B with the prompt left ends at {out.round(3)}, and the page says LEFT"
assert np.allclose(guided_sample(START_B, STEPS_10, LEFT[None], 3), LEFT, atol=1e-3), \
    "a weight of 3 still ends at LEFT, because the estimate overshoots and the last jump returns it"
assert np.allclose(guided_sample(START_B, STEPS_10, LEFT[None], 0), sample(START_B, STEPS_10), atol=1e-6), \
    "a weight of 0 is no guidance at all, so it should match Part 3"
assert np.allclose(guided_sample(START_B, STEPS_10, DATA, 3), RIGHT, atol=1e-3), \
    "a prompt that allows every training image cancels out, so guidance leaves the run alone"
assert np.allclose(guided_sample(START_A, STEPS_10, LEFT[None], 2.5, negative_images=RIGHT[None]),
                   [2.5, -1.5, 2.5, -1.5], atol=1e-6), \
    f"with the negative prompt right the run ends at {guided_sample(START_A, STEPS_10, LEFT[None], 2.5, negative_images=RIGHT[None]).round(3)}, and every step gives the same value"
print("Part 4 passed")

## Solution

### Part 1: A noise schedule for any number of steps

In [ ]:
def schedule(T):
    """The betas and the signal kept after each of T steps."""
    betas = np.linspace(1e-4, 0.02, T)
    # the signal kept after each step is the running product of the shares
    alpha_bar = np.cumprod(1 - betas)
    return betas, alpha_bar


ALPHA_BAR = schedule(1000)[1]


def add_noise(x0, t, eps):
    """The image after step t, for a clean image x0 and the noise eps."""
    kept = 1.0 if t == 0 else ALPHA_BAR[t - 1]
    return np.sqrt(kept) * x0 + np.sqrt(1 - kept) * eps

**Why it works.** `alpha_bar` is the running product of `1 - betas`, so
`np.cumprod(1 - betas)` gives the signal kept after each step, and it falls at
every step because each factor is below one. `betas` runs from `1e-4` to `0.02`
because `np.linspace` puts the two ends on the first and last step. Step 0 has
no noise added, so its signal is the whole 1 and `add_noise` has to return
`x0` there rather than reading the first entry of `alpha_bar`, which belongs to
step 1. Taking the square root of the signal and of `1 - signal` is what keeps
the two terms the same size.

### Part 2: The best possible denoiser

In [ ]:
def best_guess(xt, t, images):
    """The best possible denoiser: (estimate, weights) for the training set images."""
    kept = 1.0 if t == 0 else ALPHA_BAR[t - 1]
    d = ((xt - np.sqrt(kept) * images) ** 2).sum(axis=1) / (2 * (1 - kept))
    weights = np.exp(-(d - d.min()))
    weights = weights / weights.sum()
    return weights @ images, weights

**Why it works.** The weights are a softmax over the negative distances, and
a softmax is easiest to write by taking the smallest distance out first:
`exp(-(d - d.min()))` is the same set of numbers as `exp(-d)` up to one factor
that cancels in the sum, but it can never overflow, and with a far away image
the unshifted exponentials underflow to zero and the sum becomes 0 / 0. The
divisor `2 * (1 - alpha_bar)` is the temperature: it makes the distances grow
as the image gets noisier, which is what makes the estimate move from the average
of the training images towards one of them.

The estimate is the weights times the images, which broadcasts over the four
pixels because every weight multiplies a whole image.

### Part 3: A deterministic sampler over any steps

In [ ]:
def sample(start, steps, images=DATA):
    """The image the deterministic sampler returns after walking the steps."""
    x = np.asarray(start, dtype=float).copy()
    for t, t_next in zip(steps[:-1], steps[1:]):
        estimate, _ = best_guess(x, t, images)
        noise = (x - np.sqrt(ALPHA_BAR[t - 1]) * estimate) / np.sqrt(1 - ALPHA_BAR[t - 1])
        kept_next = 1.0 if t_next == 0 else ALPHA_BAR[t_next - 1]
        x = np.sqrt(kept_next) * estimate + np.sqrt(1 - kept_next) * noise
    return x

**Why it works.** Each pass through the loop replaces the image with one of
the same shape, using the estimate at the step it is at and the noise that
estimate implies. The noise is recomputed at every step rather than carried
along, which is what makes the sampler follow the network afresh each time.

The jump must use the signal of the step it is jumping **to**, not the step it
is at. Using `alpha_bar(t)` for both makes the new image equal the old one, so
the sampler returns its start unchanged, and a jump straight to step 0 returns
the estimate at the first step, since the noise factor there is zero.

### Part 4: Classifier-free guidance

In [ ]:
def guided_sample(start, steps, prompt_images, w, images=DATA, negative_images=DATA):
    """The image the guided sampler returns after walking the steps."""
    x = np.asarray(start, dtype=float).copy()
    for t, t_next in zip(steps[:-1], steps[1:]):
        unprompted, _ = best_guess(x, t, negative_images)
        prompted, _ = best_guess(x, t, prompt_images)
        estimate = unprompted + w * (prompted - unprompted)
        noise = (x - np.sqrt(ALPHA_BAR[t - 1]) * estimate) / np.sqrt(1 - ALPHA_BAR[t - 1])
        kept_next = 1.0 if t_next == 0 else ALPHA_BAR[t_next - 1]
        x = np.sqrt(kept_next) * estimate + np.sqrt(1 - kept_next) * noise
    return x

**Why it works.** Guidance is a straight line: start at the unprompted
estimate and move `w` times the gap between the two estimates, so `w = 0`
leaves the run alone and `w = 1` lands exactly on the prompted estimate.

A prompt that names every training image makes the two estimates equal, so the
gap is zero and guidance does nothing however large `w` is. A negative prompt
does the opposite: the unprompted estimate becomes exactly that prompt's
training image, so the gap is the whole distance from RIGHT to LEFT and every
step produces the same guided estimate, which is why the run returns it
unchanged and off the ends of the valid pixel range.